In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
DATA_PATH = "student_performance.csv"

df = pd.read_csv(DATA_PATH)

df.head(10)

,student_id,name,gender,age,class,teacher_id,subject,test1_score,test2_score,final_exam_score,final_grade_letter,attendance_percentage,study_hours_per_week,assignments_submitted,pass_fail
0,1,Carson Rivera,Female,14,9th,105,Math,23.5,40.5,50.5,F,70.2,3.1,56.7,0
1,2,Ashley Walters,Male,15,10th,107,Computer,77.7,80.6,82.3,B,85.3,6.9,81.0,1
2,3,Linda Robinson,Male,15,10th,103,English,75.0,73.4,64.5,C,83.1,4.3,66.3,1
3,4,Brandon James,Female,17,11th,117,Computer,46.7,53.1,51.2,F,74.1,0.4,44.9,0
4,5,Helen Grant,Female,16,10th,112,Science,38.0,57.3,47.6,F,73.8,1.4,49.9,0
5,6,Amy Marshall,Male,15,9th,111,Art,93.0,83.6,83.1,A,83.4,6.8,73.9,1
6,7,Michelle Watkins,Female,16,10th,109,Math,40.9,51.7,59.1,F,76.5,1.5,53.2,1
7,8,Sandra Mitchell,Female,15,10th,110,History,61.7,63.7,76.5,C,78.6,4.1,56.4,1
8,9,Sara Alvarez,Male,16,11th,105,Art,85.4,78.5,77.3,B,84.4,6.8,91.3,1
9,10,Matthew Robertson,Male,14,9th,109,History,44.8,55.3,60.8,F,79.8,2.2,54.5,1


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

Shape: (1000, 15)

Columns:
['student_id', 'name', 'gender', 'age', 'class', 'teacher_id', 'subject', 'test1_score', 'test2_score', 'final_exam_score', 'final_grade_letter', 'attendance_percentage', 'study_hours_per_week', 'assignments_submitted', 'pass_fail']

Missing values:
student_id               0
name                     0
gender                   0
age                      0
class                    0
teacher_id               0
subject                  0
test1_score              0
test2_score              0
final_exam_score         0
final_grade_letter       0
attendance_percentage    0
study_hours_per_week     0
assignments_submitted    0
pass_fail                0
dtype: int64

Duplicates: 0


In [4]:
TARGET = "pass_fail"

print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True))

pass_fail
1    830
0    170
Name: count, dtype: int64
pass_fail
1    0.83
0    0.17
Name: proportion, dtype: float64


In [5]:
drop_cols = [
    "student_id",
    "name",
    "teacher_id",
    "final_grade_letter"
]

df_model = df.drop(columns=drop_cols)

df_model.head()

,gender,age,class,subject,test1_score,test2_score,final_exam_score,attendance_percentage,study_hours_per_week,assignments_submitted,pass_fail
0,Female,14,9th,Math,23.5,40.5,50.5,70.2,3.1,56.7,0
1,Male,15,10th,Computer,77.7,80.6,82.3,85.3,6.9,81.0,1
2,Male,15,10th,English,75.0,73.4,64.5,83.1,4.3,66.3,1
3,Female,17,11th,Computer,46.7,53.1,51.2,74.1,0.4,44.9,0
4,Female,16,10th,Science,38.0,57.3,47.6,73.8,1.4,49.9,0


In [6]:
df_model["average_test_score"] = (
    df_model["test1_score"] + df_model["test2_score"]
) / 2

df_model["study_attendance"] = (
    df_model["study_hours_per_week"] *
    df_model["attendance_percentage"]
)

df_model["total_assessment_score"] = (
    df_model["test1_score"] +
    df_model["test2_score"] +
    df_model["final_exam_score"]
) / 3

df_model.head()

,gender,age,class,subject,test1_score,test2_score,final_exam_score,attendance_percentage,study_hours_per_week,assignments_submitted,pass_fail,average_test_score,study_attendance,total_assessment_score
0,Female,14,9th,Math,23.5,40.5,50.5,70.2,3.1,56.7,0,32.00,217.62,38.166667
1,Male,15,10th,Computer,77.7,80.6,82.3,85.3,6.9,81.0,1,79.15,588.57,80.200000
2,Male,15,10th,English,75.0,73.4,64.5,83.1,4.3,66.3,1,74.20,357.33,70.966667
3,Female,17,11th,Computer,46.7,53.1,51.2,74.1,0.4,44.9,0,49.90,29.64,50.333333
4,Female,16,10th,Science,38.0,57.3,47.6,73.8,1.4,49.9,0,47.65,103.32,47.633333


In [7]:
df_model["attendance_level"] = pd.cut(
    df_model["attendance_percentage"],
    bins=[0, 60, 75, 90, 100],
    labels=["Low", "Medium", "High", "Very High"],
    include_lowest=True
)

df_model["attendance_level"].value_counts()

,count
attendance_level,
High,718
Medium,154
Very High,128
Low,0


In [8]:
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 14)
y shape: (1000,)


In [9]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical:")
print(numerical_features)

print("\nCategorical:")
print(categorical_features)

Numerical:
['age', 'test1_score', 'test2_score', 'final_exam_score', 'attendance_percentage', 'study_hours_per_week', 'assignments_submitted', 'average_test_score', 'study_attendance', 'total_assessment_score']

Categorical:
['gender', 'class', 'subject', 'attendance_level']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (800, 14)
Testing: (200, 14)


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [12]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [13]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [14]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[ 34   0]
 [  0 166]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        34
           1       1.00      1.00      1.00       166

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [15]:
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = model.named_steps[
    "classifier"
].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(10)

,Feature,Importance
24,remainder__total_assessment_score,0.252214
21,remainder__assignments_submitted,0.176646
22,remainder__average_test_score,0.125446
17,remainder__test2_score,0.096295
18,remainder__final_exam_score,0.086580
16,remainder__test1_score,0.084119
19,remainder__attendance_percentage,0.063423
23,remainder__study_attendance,0.034332
13,cat__attendance_level_Medium,0.027985
20,remainder__study_hours_per_week,0.021468


In [16]:
df_model["exam_improvement"] = (
    df_model["final_exam_score"] - df_model["test1_score"]
)

print(df_model[
    ["test1_score", "final_exam_score", "exam_improvement"]
].head())

   test1_score  final_exam_score  exam_improvement
0         23.5              50.5              27.0
1         77.7              82.3               4.6
2         75.0              64.5             -10.5
3         46.7              51.2               4.5
4         38.0              47.6               9.6


Task 1: I created a new feature called exam_improvement, which represents the difference between the final exam score and the first test score. This feature may help predict pass_fail because it captures whether a student's academic performance improved or declined over time.

In [17]:
df_model["attendance_level_new"] = pd.cut(
    df_model["attendance_percentage"],
    bins=[0, 70, 85, 100],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

print(df_model["attendance_level_new"].value_counts())

attendance_level_new
Medium    625
High      334
Low        41
Name: count, dtype: int64


Task 2: I changed the attendance grouping rule from four categories to three categories: Low, Medium, and High. This provides a simpler representation of attendance behavior. The effect on model performance can be evaluated by retraining the model and comparing the accuracy.

In [18]:
for bins in [3, 5, 7]:
    temp = pd.cut(
        df_model["study_hours_per_week"],
        bins=bins
    )

    print("\nNumber of bins:", bins)
    print(temp.value_counts().sort_index())


Number of bins: 3
study_hours_per_week
(-0.012, 4.0]    488
(4.0, 8.0]       459
(8.0, 12.0]       53
Name: count, dtype: int64

Number of bins: 5
study_hours_per_week
(-0.012, 2.4]    231
(2.4, 4.8]       379
(4.8, 7.2]       278
(7.2, 9.6]        94
(9.6, 12.0]       18
Name: count, dtype: int64

Number of bins: 7
study_hours_per_week
(-0.012, 1.714]    143
(1.714, 3.429]     258
(3.429, 5.143]     248
(5.143, 6.857]     213
(6.857, 8.571]     107
(8.571, 10.286]     23
(10.286, 12.0]       8
Name: count, dtype: int64


Task 3: I experimented with 3, 5, and 7 bins for study_hours_per_week. Using fewer bins produces broader groups, while more bins provide more detailed categories. Different grouping levels can affect the information available to the classifier and therefore may influence model performance and feature importance.

In [19]:
print("Top 10 Most Important Features:")
print(feature_importance.head(10))

selected_features = feature_importance.head(10)

print("\nSelected Features:")
print(selected_features)

Top 10 Most Important Features:
                              Feature  Importance
24  remainder__total_assessment_score    0.252214
21   remainder__assignments_submitted    0.176646
22      remainder__average_test_score    0.125446
17             remainder__test2_score    0.096295
18        remainder__final_exam_score    0.086580
16             remainder__test1_score    0.084119
19   remainder__attendance_percentage    0.063423
23        remainder__study_attendance    0.034332
13       cat__attendance_level_Medium    0.027985
20    remainder__study_hours_per_week    0.021468

Selected Features:
                              Feature  Importance
24  remainder__total_assessment_score    0.252214
21   remainder__assignments_submitted    0.176646
22      remainder__average_test_score    0.125446
17             remainder__test2_score    0.096295
18        remainder__final_exam_score    0.086580
16             remainder__test1_score    0.084119
19   remainder__attendance_percentage    0.06342

Task 4: Feature selection was performed using the feature importance scores from the Random Forest model. The top 10 features were selected as the most influential features. Feature selection can be useful because it reduces less important information and makes the model easier to interpret.